In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# EXP09 — Advanced GBDT Ensemble: Stacking + Horizon-wise Blending + High-Wave Boosting

EXP05 (LightGBM), EXP06 (XGBoost), EXP07 (CatBoost)의 OOF 예측치와 테스트 제출 파일을 기반으로 고도화된 앙상블 및 후처리를 수행합니다.

### 파이프라인 구성
1. **OOF 파일 로드**: `oof_exp05.csv`, `oof_exp06.csv`, `oof_exp07.csv`
2. **단일 모델 OOF 성능 벤치마크** ($h_s \ge 1.5\text{m}$ 고파랑 통합 RMSE)
3. **앙상블 기법 1 — 리드타임별 가중치 최적화 (SLSQP)**
4. **앙상블 기법 2 — 메타 모델 2차 스태킹 (Non-negative Ridge Regression)**
5. **고파랑($h_s \ge 1.5\text{m}$) 맞춤형 부스팅 계수($\alpha$) 탐색** (Shrinkage 보정)
6. **최종 앙상블 OOF (`oof_exp09.csv`) 및 제출 파일 (`submission_exp09.csv`) 생성**

In [1]:
# ============================================================
# 0. SETUP & PACKAGES
# ============================================================
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print("Setup Complete for EXP09 Advanced Ensemble.")

Setup Complete for EXP09 Advanced Ensemble.


In [2]:
# ============================================================
# 1. CONFIG & FILE PATHS
# ============================================================
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v3.csv")
TEST_INDEX_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_index.csv")

# OOF 파일 경로
OOF_EXP05_PATH = Path(PROJECT_ROOT / "oof" / "oof_exp05.csv")
OOF_EXP06_PATH = Path(PROJECT_ROOT / "oof" / "oof_exp06.csv")
OOF_EXP07_PATH = Path(PROJECT_ROOT / "oof" / "oof_exp07.csv")

# Submission 파일 경로
SUB_EXP05_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp05.csv")
SUB_EXP06_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp06.csv")
SUB_EXP07_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp07.csv")

OOF_OUTPUT_PATH = Path(PROJECT_ROOT / "oof" / "oof_exp09.csv")
SUBMISSION_OUTPUT_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp09.csv")

LEAD_HOURS = [3, 6, 9, 12, 18, 24]
STEPS_PER_HOUR = 6
STEP_MINUTES = 10
INPUT_LEN = 289
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]

print("Target Leads:", LEAD_HOURS)

Target Leads: [3, 6, 9, 12, 18, 24]


In [ ]:
# ============================================================
# 2. LOAD OOF PREDICTIONS & GROUND TRUTH
# ============================================================
oof_lgb = pd.read_csv(OOF_EXP05_PATH)
oof_xgb = pd.read_csv(OOF_EXP06_PATH)
oof_cat = pd.read_csv(OOF_EXP07_PATH)

# 원본 Ground Truth 재생성
df = pd.read_csv(DATA_PATH, parse_dates=["time"])
df = df.sort_values(["station", "time"]).reset_index(drop=True)
if "hs_original_observed" not in df.columns:
    df["hs_original_observed"] = df["hs"].notna().astype(np.int8)

y_rows, origin_hs_list = [], []
max_lead = max(LEAD_STEPS)
STRIDE = 1

for st, g in df.groupby("station"):
    g = g.sort_values("time").reset_index(drop=True)
    dt = pd.Series(g["time"]).diff().dt.total_seconds().div(60).to_numpy()

    for end_idx in range(INPUT_LEN - 1, len(g) - max_lead, STRIDE):
        start_idx = end_idx - INPUT_LEN + 1
        if not np.all(dt[start_idx + 1:end_idx + 1] == STEP_MINUTES):
            continue

        target_idx = [end_idx + s for s in LEAD_STEPS]
        y = g.loc[target_idx, "hs"].to_numpy(dtype=np.float32)
        obs = g.loc[target_idx, "hs_original_observed"].to_numpy(dtype=np.int8)
        if not np.isfinite(y).all() or not np.all(obs == 1):
            continue

        y_rows.append(y)
        origin_hs_list.append(float(g.loc[end_idx, "hs"]))

Y_true_arr = np.asarray(y_rows, dtype=np.float32)
origin_hs_arr = np.asarray(origin_hs_list, dtype=np.float32)
high_mask = origin_hs_arr >= 1.5
sample_weights = np.where(high_mask, 3.33, 1.0)

print(f"Loaded OOF Samples: {len(oof_lgb)} rows")
print(f"High-Wave Samples (hs >= 1.5m): {high_mask.sum()} / {len(origin_hs_arr)}")

Loaded OOF Samples: 43200 rows
High-Wave Samples (hs >= 1.5m): 10396 / 43200


In [4]:
# ============================================================
# 3. SINGLE MODEL PERFORMANCE BENCHMARK
# ============================================================
def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def get_metrics(name, preds):
    high_true = Y_true_arr[high_mask]
    high_pred = preds[high_mask]
    res = {
        "Model": name,
        "Comp_Integrated_RMSE": calc_rmse(high_true, high_pred),
        "Overall_RMSE": calc_rmse(Y_true_arr, preds)
    }
    for idx, h in enumerate(LEAD_HOURS):
        res[f"Comp_{h}h"] = calc_rmse(high_true[:, idx], high_pred[:, idx])
    return res

p_lgb = oof_lgb[[f"pred_{h}h" for h in LEAD_HOURS]].to_numpy()
p_xgb = oof_xgb[[f"pred_{h}h" for h in LEAD_HOURS]].to_numpy()
p_cat = oof_cat[[f"pred_{h}h" for h in LEAD_HOURS]].to_numpy()

bm_df = pd.DataFrame([
    get_metrics("EXP05 (LightGBM)", p_lgb),
    get_metrics("EXP06 (XGBoost)", p_xgb),
    get_metrics("EXP07 (CatBoost)", p_cat)
])
display(bm_df)

,Model,Comp_Integrated_RMSE,Overall_RMSE,Comp_3h,Comp_6h,Comp_9h,Comp_12h,Comp_18h,Comp_24h
0,EXP05 (LightGBM),0.240628,0.187852,0.243416,0.244565,0.240138,0.247374,0.231515,0.236405
1,EXP06 (XGBoost),0.240795,0.186909,0.240583,0.241678,0.242172,0.246497,0.234266,0.239411
2,EXP07 (CatBoost),0.278852,0.281843,0.278654,0.278368,0.280989,0.278632,0.274263,0.282143


In [5]:
# ============================================================
# 4. COMPARE ENSEMBLE METHODS: SLSQP vs NON-NEGATIVE RIDGE
# ============================================================
# Method A: SLSQP Horizon-wise Weights
slsqp_weights = {}
oof_slsqp = np.zeros_like(Y_true_arr)

for idx, h in enumerate(LEAD_HOURS):
    c_lgb = p_lgb[:, idx]
    c_xgb = p_xgb[:, idx]
    c_cat = p_cat[:, idx]
    y_tr = Y_true_arr[:, idx]
    
    def obj(w):
        pred = w[0] * c_lgb + w[1] * c_xgb + w[2] * c_cat
        return calc_rmse(y_tr[high_mask], pred[high_mask])
    
    res = minimize(obj, [1/3, 1/3, 1/3], method='SLSQP',
                   bounds=[(0, 1), (0, 1), (0, 1)],
                   constraints=({'type': 'eq', 'fun': lambda w: sum(w) - 1.0}))
    slsqp_weights[h] = res.x
    oof_slsqp[:, idx] = res.x[0] * c_lgb + res.x[1] * c_xgb + res.x[2] * c_cat

# Method B: Non-negative Stacking Ridge
ridge_models = {}
oof_ridge = np.zeros_like(Y_true_arr)

for idx, h in enumerate(LEAD_HOURS):
    X_meta = np.column_stack([p_lgb[:, idx], p_xgb[:, idx], p_cat[:, idx]])
    y_tr = Y_true_arr[:, idx]
    
    # Non-negative constraint Ridge with high-wave sample weights
    ridge = Ridge(alpha=5.0, positive=True, fit_intercept=True)
    ridge.fit(X_meta, y_tr, sample_weight=sample_weights)
    ridge_models[h] = ridge
    oof_ridge[:, idx] = ridge.predict(X_meta)

comp_summary = pd.DataFrame([
    get_metrics("Simple Average (1:1:1)", (p_lgb + p_xgb + p_cat) / 3.0),
    get_metrics("Method A (SLSQP Optimal Weights)", oof_slsqp),
    get_metrics("Method B (Stacking Ridge)", oof_ridge)
])
display(comp_summary)

,Model,Comp_Integrated_RMSE,Overall_RMSE,Comp_3h,Comp_6h,Comp_9h,Comp_12h,Comp_18h,Comp_24h
0,Simple Average (1:1:1),0.240237,0.204815,0.245125,0.243314,0.241464,0.243070,0.230838,0.237319
1,Method A (SLSQP Optimal Weights),0.234339,0.183418,0.237770,0.237806,0.234806,0.239744,0.225141,0.230443
2,Method B (Stacking Ridge),0.234107,0.188773,0.240034,0.239715,0.236116,0.237472,0.222859,0.227919


In [6]:
# ============================================================
# 5. HIGH-WAVE BOOST FACTOR (ALPHA) OPTIMIZATION
# ============================================================
# 더 우수한 앙상블 베이스라인 선택
score_a = calc_rmse(Y_true_arr[high_mask], oof_slsqp[high_mask])
score_b = calc_rmse(Y_true_arr[high_mask], oof_ridge[high_mask])
base_oof = oof_slsqp.copy() if score_a <= score_b else oof_ridge.copy()
selected_method = "SLSQP" if score_a <= score_b else "Ridge"
print(f"Selected Base Ensemble: {selected_method} (Comp RMSE: {min(score_a, score_b):.5f})")

# 리드타임별 고파랑 부스팅 계수 탐색
best_alphas = {}
final_boosted_oof = base_oof.copy()

for idx, h in enumerate(LEAD_HOURS):
    p_lead = base_oof[:, idx]
    y_lead = Y_true_arr[:, idx]
    
    def boost_obj(alpha):
        p_mod = p_lead.copy()
        p_mod[high_mask] = p_mod[high_mask] * alpha[0]
        return calc_rmse(y_lead[high_mask], p_mod[high_mask])
    
    res_a = minimize(boost_obj, [1.00], bounds=[(0.95, 1.08)], method='SLSQP')
    best_alphas[h] = float(res_a.x[0])
    final_boosted_oof[high_mask, idx] = final_boosted_oof[high_mask, idx] * best_alphas[h]
    print(f"Lead +{h:02d}h -> Optimal Alpha: {best_alphas[h]:.4f} | Boosted RMSE: {res_a.fun:.5f}")

comp_boosted = calc_rmse(Y_true_arr[high_mask], final_boosted_oof[high_mask])
print("\n" + "=" * 75)
print(f"★ EXP09 FINAL POST-PROCESSED COMP RMSE: {comp_boosted:.5f} m")
print("=" * 75)

Selected Base Ensemble: Ridge (Comp RMSE: 0.23411)
Lead +03h -> Optimal Alpha: 1.0000 | Boosted RMSE: 0.24003
Lead +06h -> Optimal Alpha: 1.0000 | Boosted RMSE: 0.23971
Lead +09h -> Optimal Alpha: 1.0000 | Boosted RMSE: 0.23612
Lead +12h -> Optimal Alpha: 1.0000 | Boosted RMSE: 0.23747
Lead +18h -> Optimal Alpha: 1.0000 | Boosted RMSE: 0.22286
Lead +24h -> Optimal Alpha: 1.0000 | Boosted RMSE: 0.22792

★ EXP09 FINAL POST-PROCESSED COMP RMSE: 0.23411 m


In [7]:
# ============================================================
# 6. SAVE EXP09 OOF PREDICTIONS
# ============================================================
oof_exp09_df = pd.DataFrame(final_boosted_oof, columns=[f"pred_{h}h" for h in LEAD_HOURS])
oof_exp09_df["origin_hs"] = origin_hs_arr
oof_exp09_df["origin_time"] = oof_lgb["origin_time"]

oof_exp09_df.to_csv(OOF_OUTPUT_PATH, index=False)
print(f"Successfully Saved: {OOF_OUTPUT_PATH}")

Successfully Saved: oof_exp09.csv


In [8]:
# ============================================================
# 7. APPLY ADVANCED ENSEMBLE TO TEST SUBMISSIONS (EXP09)
# ============================================================
print("Applying Optimal Ensemble & Boosting to Test Submissions...")

sub_lgb = pd.read_csv(SUB_EXP05_PATH)
sub_xgb = pd.read_csv(SUB_EXP06_PATH)
sub_cat = pd.read_csv(SUB_EXP07_PATH)

sub_final = sub_lgb.copy()
sub_final["hs_pred"] = 0.0

for idx, h in enumerate(LEAD_HOURS):
    mask = sub_final["lead_h"] == h
    
    if selected_method == "SLSQP":
        w1, w2, w3 = slsqp_weights[h]
        pred_test = (
            w1 * sub_lgb.loc[mask, "hs_pred"] +
            w2 * sub_xgb.loc[mask, "hs_pred"] +
            w3 * sub_cat.loc[mask, "hs_pred"]
        )
    else:
        X_test_meta = np.column_stack([
            sub_lgb.loc[mask, "hs_pred"],
            sub_xgb.loc[mask, "hs_pred"],
            sub_cat.loc[mask, "hs_pred"]
        ])
        pred_test = ridge_models[h].predict(X_test_meta)
    
    # 고파랑 부스팅 계수 반영 (일괄 적용)
    pred_test = pred_test * best_alphas[h]
    sub_final.loc[mask, "hs_pred"] = pred_test

# 물리적 상/하한 클리핑
sub_final["hs_pred"] = sub_final["hs_pred"].clip(lower=0.05, upper=30.0)
sub_final.to_csv(SUBMISSION_OUTPUT_PATH, index=False, encoding="utf-8")

print("=" * 80)
print(f"EXP09 FINAL SUBMISSION SAVED: {SUBMISSION_OUTPUT_PATH}")
print("=" * 80)
print("\nPrediction Stats:")
display(sub_final["hs_pred"].describe())
display(sub_final.head(12))

Applying Optimal Ensemble & Boosting to Test Submissions...
EXP09 FINAL SUBMISSION SAVED: submission_exp09.csv

Prediction Stats:


count    1200.000000
mean        1.610312
std         0.497747
min         0.367731
25%         1.267867
50%         1.570575
75%         1.875723
max         4.105773
Name: hs_pred, dtype: float64

,case_id,station,lead_h,hs_pred
0,C0001,S-ORS,3,2.576932
1,C0001,S-ORS,6,1.595246
2,C0001,S-ORS,9,1.384983
3,C0001,S-ORS,12,1.697172
4,C0001,S-ORS,18,0.989447
5,C0001,S-ORS,24,1.195705
6,C0002,I-ORS,3,1.593999
7,C0002,I-ORS,6,1.595019
8,C0002,I-ORS,9,1.595740
9,C0002,I-ORS,12,1.634363
